# 1. Importación

Carga de las librerías necesarias y del dataset de features resultante de `02D_fe_listings_full.ipynb` (`data/processed/listings_full_features.csv`).

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/listings_full_features.csv")
df.shape

(13258, 73)

## 2. Preparación de X e y

Separar identificadores, objetivo (`price`) y features. Convertir las columnas booleanas a 0/1 y decidir qué hacer con los nulos que quedan (`review_scores_rating`, `listing_age_days`, `days_since_last_review`), ya que una regresión lineal no admite `NaN` directamente.

### 2.1 Identificadores, objetivo y features

Además de los identificadores, hay que excluir de `feature_cols` las columnas que están calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): si se dejan dentro, el modelo no aprende ningún patrón real, simplemente deshace la fórmula y "adivina" el precio exacto. Es fuga de información, y aquí se detecta enseguida porque un modelo con fuga da un R² sospechosamente perfecto.

**Aviso pendiente**: `neighbourhood_price_encoded` (creada en `02D`) también tiene un problema parecido, aunque más leve: se calculó usando todo el dataset, no solo lo que aquí es train. El efecto es pequeño (la media de `price` con todos los datos y solo con train difieren en ~2€ de 240€, menos de un 1%), así que se deja así por ahora y se documenta como algo a corregir antes de dar cualquier modelo por definitivo, en vez de bloquear el baseline por esto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((13258, 66), (13258,))

### 2.2 Booleanas a 0/1

37 de las 69 features son booleanas (los one-hot y los flags). `scikit-learn` las admite tal cual, pero se convierten a `int` de forma explícita para evitar sorpresas.

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

37

### 2.3 Nulos restantes

`review_scores_rating`, `listing_age_days` y `days_since_last_review` son `NaN` en las 2927 filas sin reviews todavía (`has_reviews == False`). Una regresión lineal no admite `NaN`, así que para este baseline se imputan con la mediana: es una simplificación, no la solución definitiva (un modelo de árboles en `03_model_training.ipynb` podría trabajar con el `NaN` directamente), pero para un primer modelo simple es razonable, y `has_reviews` sigue disponible como columna aparte para que el modelo distinga estos casos.

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 69 columnas numéricas sin nulos, e `y` es `price` sin transformar (el logaritmo se aplica más adelante, solo para el modelo de la sección 6, no aquí en la preparación general).

## 3. Train/test split

Reservar un conjunto de test antes de tocar nada más, y guardarlo en `data/processed/` para que `03_model_training.ipynb` y `03_model_evaluation.ipynb` partan del mismo split y los resultados sean comparables entre notebooks.

### 3.1 Dividir

80/20, con `random_state` fijo para que el split sea reproducible entre notebooks. Se pasan `X`, `y` y `df` juntos a `train_test_split`, así los tres quedan divididos con exactamente el mismo reparto de filas en una sola llamada.

In [6]:
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((10606, 66), (2652, 66))

### 3.2 Guardar el split

Se guarda `df_train`/`df_test` (antes de la imputación y la conversión de booleanas de la sección 2), no `X_train`/`X_test` ya preparados. Así `03_model_training.ipynb` y `03_model_evaluation.ipynb` parten de los datos en crudo y pueden decidir su propio tratamiento de nulos (p. ej. un modelo de árboles no necesita la imputación por mediana que se hizo aquí), en vez de heredar una decisión pensada solo para la regresión lineal de este notebook.

In [7]:
df_train.to_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/listings_train.csv", index=False)
df_test.to_csv("/Users/yagocoll/Documents/Master/Airbnb/data/processed/listings_test.csv", index=False)

## 4. Baseline ingenuo

Un modelo trivial (predecir siempre la mediana de `price`, o la mediana por barrio) como suelo mínimo: cualquier modelo real tiene que superar esto para que merezca la pena.

### 4.1 Predecir siempre la media

Este es el baseline "de referencia formal": por definición, un modelo que siempre predice la media del train tiene R² ≈ 0 sobre el test. Sirve como punto cero, no como algo que se espera batir por poco. Las métricas se calculan aquí a mano con las funciones de `sklearn` ya importadas; en la sección 5 se empaquetan en una función reutilizable.

In [8]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 358.7906256660401
MAE: 160.37027603118298
R2: -0.0007477532712980572


### 4.2 Predecir siempre la mediana

Dado el sesgo de `price` (visto en la EDA), la mediana debería ser un mejor "valor típico" que la media. Se comprueba.

In [9]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 365.77343367813194
MAE: 152.8929939668175
R2: -0.040080052842706104


El MAE mejora (152.9 frente a 160.4 con la media): la mediana es más robusta a los precios extremos. Pero el R² empeora, incluso se vuelve negativo (-0.04 frente a ~0.00). No es una contradicción: el R² compara contra la media por definición, así que cualquier predicción distinta de la media puede bajarlo aunque sea mejor en otros términos. Es un recordatorio de que **la métrica que se elige como referencia importa**, no solo el modelo.

### 4.3 Predecir la mediana según `bedrooms`

`bedrooms` fue la variable más relacionada con `price` de toda la EDA (correlación 0.66). Un baseline algo menos ingenuo: en vez de una única mediana para todo el dataset, una mediana por cada valor de `bedrooms`, calculada solo con el train.

In [10]:
train_medians_by_bedrooms = X_train.assign(price=y_train).groupby("bedrooms")["price"].median()

pred_bedrooms = X_test["bedrooms"].map(train_medians_by_bedrooms).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_bedrooms)))
print("MAE:", mean_absolute_error(y_test, pred_bedrooms))
print("R2:", r2_score(y_test, pred_bedrooms))

RMSE: 264.03628338338467
MAE: 120.91823529411766
R2: 0.4580369593277013


Mejora clara en todo: RMSE de 366 a 264, MAE de 153 a 121, R² de -0.04 a **0.46**. Sin entrenar ningún modelo, solo agrupando por una variable, ya se explica cerca de la mitad de la varianza de `price`. Esto pone el listón real para la regresión lineal de la sección 6 y, sobre todo, para los modelos de `03_model_training.ipynb`: si no superan claramente este 0.46, no está aportando nada usar algo más complejo que "mirar cuántos dormitorios tiene".

## 5. Métricas de evaluación

Definir aquí las métricas que se van a usar de forma consistente en todo el modelado (RMSE, MAE, R², MAPE), para no repetir la definición en cada notebook y poder comparar baseline, entrenamiento y evaluación con la misma vara de medir.

### 5.1 Función `evaluate`

RMSE, MAE y R² ya se usaron sueltos en la sección 4. Se añade una cuarta métrica, MAPE (Mean Absolute Percentage Error): el error medio en porcentaje sobre el precio real, en vez de en euros. La ventaja es que se interpreta igual de fácil sin importar la escala del precio; la desventaja se ve enseguida al calcularla. Todo se empaqueta en una función para no repetir el código en cada modelo.

In [11]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

Se aplica la función a los tres baselines de la sección 4, para tenerlos todos juntos en una tabla. Esta tabla (`results`) se irá ampliando con cada modelo nuevo que se entrene en el notebook, empezando por la regresión lineal de la sección 6.

In [12]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_bedrooms, "Mediana por bedrooms"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,358.79,160.37,-0.00,134.43
Mediana,365.77,152.89,-0.04,96.80
Mediana por bedrooms,264.04,120.92,0.46,88.00


El MAPE sale muy alto (88-134%), más de lo que parecería razonable a simple vista. La razón es que hay anuncios muy baratos (`price` mínimo 3.5€) donde un error de pocos euros ya es un porcentaje enorme: fallar por 10€ en un anuncio de 15€ es un 67% de error, aunque en términos absolutos sea un fallo pequeño. Por eso el MAPE puede ser engañoso cuando hay valores muy bajos en los datos, y aquí conviene fiarse más de RMSE/MAE/R² que del MAPE por sí solo. Se deja calculado igualmente, porque en otros contextos (o si se filtran los anuncios más baratos) sí puede ser útil.

## 6. Baseline real: regresión lineal

Un primer modelo simple e interpretable, entrenado sobre `price_log` por el sesgo ya visto en la EDA. Sirve de referencia real antes de pasar a modelos más complejos en `03_model_training.ipynb`.

### 6.1 Entrenar

Se entrena sobre `log1p(price)`, no sobre `price` directamente, por el sesgo ya visto en la EDA. Las predicciones se deshacen con `expm1` antes de evaluar, para comparar en euros contra los mismos `y_test` que se usaron en los baselines ingenuos.

In [13]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [14]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,358.79,160.37,-0.00,134.43
Mediana,365.77,152.89,-0.04,96.80
Mediana por bedrooms,264.04,120.92,0.46,88.00
Regresión lineal (log),173.47,66.89,0.77,26.77


**R²=0.77, MAE≈67€**, muy por encima del mejor baseline ingenuo (R²=0.46, MAE≈121€). Con 66 features y un modelo real, se explica más del triple de varianza extra que con solo `bedrooms`. Es la primera prueba de que el resto de variables (barrio, `has_license`, tipo de propiedad...) también aportan, no solo el tamaño.

### 6.3 Un aviso a tener en cuenta

Al entrenar aparecen avisos de `numpy` (`divide by zero`, `overflow`... `encountered in matmul`). Antes de ignorarlos sin más, conviene entender de dónde vienen.

In [15]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(2.162186680732821e+19)

Un número de condición de ese tamaño (billones de veces mayor que 1) indica una matriz muy mal condicionada: hay combinaciones de columnas casi redundantes entre sí. Tiene sentido, porque `X` incluye a propósito tanto la versión bruta como la versión `_log` de varias variables (`bedrooms`/`bedrooms_log`, `bathrooms`/`bathrooms_log`...) y varias codificaciones categóricas que se solapan (`room_type_Entire home/apt` correlaciona 0.96 con `host_entire_homes_ratio`, por ejemplo).

Esto **no invalida las métricas**: `scikit-learn` resuelve la regresión con SVD, que sigue dando una solución válida incluso con columnas casi redundantes, y las predicciones no tienen ningún `NaN` ni `inf`. Lo que sí invalida es **interpretar los coeficientes uno a uno** ("esta variable pesa más que esta otra"): con tanta redundancia, el peso se puede repartir de forma casi arbitraria entre columnas que miden casi lo mismo. Para eso haría falta o una selección de variables más cuidada (quedarse con una sola versión de cada par bruta/log), o un modelo regularizado (Ridge/Lasso), que se deja para `03_model_training.ipynb`.

## 7. Conclusiones

Resumen de los resultados del baseline: qué error se puede esperar como mínimo, qué variables parecen pesar más en un modelo simple, y qué se espera mejorar con un modelo más complejo.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 358.79 | 160.37 | -0.00 | 134.43 |
| Mediana | 365.77 | 152.89 | -0.04 | 96.80 |
| Mediana por `bedrooms` | 264.04 | 120.92 | 0.46 | 88.00 |
| Regresión lineal (log) | 173.47 | 66.89 | 0.77 | 26.77 |

Cada paso mejora sobre el anterior: pasar de un número fijo a agrupar por `bedrooms` ya explica casi la mitad de la varianza sin entrenar nada; un modelo real con las 66 variables sube el R² a 0.77 y baja el error medio a unos 67€.

### Dos problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night` estaban calculadas a partir de `price` y se habían colado como features. Daban un R²=1.0 sospechoso, se detectaron y se excluyeron en la sección 2.
- **Multicolinealidad severa** en la regresión lineal (número de condición ~2×10¹⁹), por tener a la vez la versión bruta y `_log` de varias variables y codificaciones categóricas solapadas. No afecta a las métricas de predicción, pero sí impide interpretar los coeficientes uno a uno con este modelo tal cual está planteado.
- Queda además documentada, sin corregir todavía, la fuga más leve de `neighbourhood_price_encoded` (calculada con todo el dataset en `02D`, no solo con train). Su impacto estimado es pequeño (~1%).

### El listón para `03_model_training.ipynb`

Cualquier modelo más complejo (Random Forest, XGBoost, LightGBM...) tiene que superar claramente **R²=0.77 / MAE≈67€** para justificar la complejidad añadida. Además, un modelo de árboles no necesita la imputación por mediana de la sección 2.3 (puede trabajar con los `NaN` de `review_scores_rating` directamente) ni le afecta la multicolinealidad de la sección 6.3, así que puede aprovechar mejor tener ambas versiones (bruta y log) de las variables sin ese coste.

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/`, con el mismo split (80/20, `random_state=42`) para que `03_model_training.ipynb` y `03_model_evaluation.ipynb` trabajen sobre las mismas filas y los resultados sean comparables entre notebooks.